In [70]:
import pandas as pd #
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
from scipy.stats import zscore

csv_path1= "./dataMA/Landslide Logger - Data.csv"
csv_path2 = "./dataMA/monitoring_data_2025-11-28_13-27-14.csv"
#device = "103"
#save_path = f"./data/dev{device}_prepared.csv"


df1 = pd.read_csv(csv_path1)
df2 = pd.read_csv(csv_path1)
display(df1.columns)
display(df2.columns)

C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\55951582.py:14: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(csv_path1)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\55951582.py:15: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(csv_path1)


Index(['timestamp', 'devID', 'soil', 'rain', 'temp', 'humi', 'geo', 'lat',
       'lng', 'iso_score', 'iso_risk', 'lstm_score', 'lstm_risk', 'confidence',
       '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14'],
      dtype='object')

Index(['timestamp', 'devID', 'soil', 'rain', 'temp', 'humi', 'geo', 'lat',
       'lng', 'iso_score', 'iso_risk', 'lstm_score', 'lstm_risk', 'confidence',
       '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14'],
      dtype='object')

In [71]:
df = pd.concat([df1, df2], ignore_index=True)

display(df.columns)

Index(['timestamp', 'devID', 'soil', 'rain', 'temp', 'humi', 'geo', 'lat',
       'lng', 'iso_score', 'iso_risk', 'lstm_score', 'lstm_risk', 'confidence',
       '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14'],
      dtype='object')

In [72]:
df.drop(columns=['iso_score', 'lstm_score', 'confidence',
       '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14'], inplace=True)
df.columns

Index(['timestamp', 'devID', 'soil', 'rain', 'temp', 'humi', 'geo', 'lat',
       'lng', 'iso_risk', 'lstm_risk'],
      dtype='object')

In [73]:

#for col in ['soil', 'rain', 'temp', 'humi', 'geo', 'lat', 'lng', 'iso_score', 'iso_risk', 'lstm_score', 'lstm_risk', 'confidence', 'rf_risk', 'devID']:
for col in ['soil', 'rain', 'temp', 'humi', 'geo', 'devID']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df

#df = df[~df['timestamp'].str.contains(r'\(ICT\)', na=False)]
#df['timestamp'] = df['timestamp'].str.replace(r'\(ICT\)', '', regex=True).str.strip()

#def clean_and_format_timestamp(val):
#    if pd.isnull(val):
#        return None
#    val = str(val).replace('(ICT)', '').strip()
#    try:
#        dt = pd.to_datetime(val)
#        return dt.isoformat()
#    except Exception:
#        return None  
#
#df['timestamp'] = df['timestamp'].apply(clean_and_format_timestamp)

df.columns = df.columns.str.strip()
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'geo'])

#df = df.drop(columns=['Unnamed: 15', 'confidence','rf_risk'], errors='ignore')
#df = df[(df['lat'] > 1.0) & (df['lat'] < 8000.0)]
#df = df[df['lng'] > 15.0]
df = df[df['soil'] > 0]
df = df[df['rain'] >= 0]
df = df[df['humi'] > 0]
#df = df[(df['devID'] == 104) | (df['devID'] == 103)]
df = df[(df['geo'] >= 0) & (df['geo'] < 40)]
df = df[df['temp'] > 0]
df = df[(df['devID'] > 0) & (df['devID'] < 1000)]
#df = df[df['iso_score'] > -85000]
#df = df[df['lstm_score'] >= 0]
#df = df[df['timestamp'] > '3035-09-11']

#df = df[df['devID'] == int(device)]
df

,timestamp,devID,soil,rain,temp,humi,geo,lat,lng,iso_risk,lstm_risk
2,2025-11-01 06:35:13.489,108.0,0.798555,0.0,34.049999,66.650002,0.000000,-1.000000,-1.000000,0,0
5,2025-11-01 06:36:31.666,108.0,0.534899,0.0,34.233334,67.233335,0.333333,-1.000000,-1.000000,0,0
9,2025-11-01 06:37:51.586,108.0,1.604698,0.0,34.250000,65.700001,1.000000,-1.000000,-1.000000,0,0
10,2025-11-01 06:42:51.995,108.0,1.604698,0.0,34.299999,61.299999,1.000000,8.873381,99.764809,0,0
11,2025-11-01 06:43:52.076,108.0,1.604698,0.0,34.500000,60.700001,1.000000,8.872817,99.762062,0,0
...,...,...,...,...,...,...,...,...,...,...,...
169153,2025-11-27 16:36:31.541,106.0,46.400002,0.0,22.600000,87.699997,0.277745,8.886664,99.784554,0,0
169154,2025-11-27 16:37:16.605,106.0,46.400002,0.0,22.700001,87.099998,0.233911,8.886664,99.784554,0,0
169155,2025-11-27 16:38:01.694,106.0,46.400002,0.0,22.750000,86.700001,0.237196,8.886664,99.784554,0,0
169156,2025-11-27 16:38:46.754,106.0,46.400002,0.0,22.700001,89.000000,0.258948,8.886662,99.784546,0,0


In [74]:
df.describe()

,timestamp,devID,soil,rain,temp,humi,geo,lat,lng,iso_risk,lstm_risk
count,168276,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000
mean,2025-11-10 17:37:29.208119552,107.126827,38.479539,0.412447,26.830935,85.649979,5.802132,9.316622,102.539911,0.079940,0.045128
min,2025-11-01 06:35:13.489000,105.000000,0.534899,0.000000,16.450001,33.750000,0.000000,-1.000000,-1.000000,0.000000,0.000000
25%,2025-11-06 10:39:33.303000064,106.000000,34.299999,0.000000,23.849999,77.599998,0.385413,8.866824,99.749580,0.000000,0.000000
50%,2025-11-10 12:13:51.344499968,107.000000,37.700001,0.000000,25.200001,89.750000,0.612796,8.868599,99.754539,0.000000,0.000000
75%,2025-11-14 04:00:50.000999936,109.000000,43.000000,0.000000,29.900000,95.799999,9.768927,8.886636,99.784538,0.000000,0.000000
max,2025-11-27 16:39:31.772000,109.000000,64.300003,45.681900,38.400002,100.000000,39.624933,8138.283203,23593.232420,2.000000,2.000000
std,NaN,1.442573,6.900702,2.418531,3.844727,12.438254,8.525796,58.061751,264.857615,0.346784,0.278708


In [75]:
df.columns = df.columns.str.strip()
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'geo'])  # ต้องมี timestamp และ geo

for col in ['soil', 'rain', 'temp', 'humi', 'geo', 'lat', 'lng', 'iso_risk', 'lstm_risk']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df

,timestamp,devID,soil,rain,temp,humi,geo,lat,lng,iso_risk,lstm_risk
2,2025-11-01 06:35:13.489,108.0,0.798555,0.0,34.049999,66.650002,0.000000,-1.000000,-1.000000,0,0
5,2025-11-01 06:36:31.666,108.0,0.534899,0.0,34.233334,67.233335,0.333333,-1.000000,-1.000000,0,0
9,2025-11-01 06:37:51.586,108.0,1.604698,0.0,34.250000,65.700001,1.000000,-1.000000,-1.000000,0,0
10,2025-11-01 06:42:51.995,108.0,1.604698,0.0,34.299999,61.299999,1.000000,8.873381,99.764809,0,0
11,2025-11-01 06:43:52.076,108.0,1.604698,0.0,34.500000,60.700001,1.000000,8.872817,99.762062,0,0
...,...,...,...,...,...,...,...,...,...,...,...
169153,2025-11-27 16:36:31.541,106.0,46.400002,0.0,22.600000,87.699997,0.277745,8.886664,99.784554,0,0
169154,2025-11-27 16:37:16.605,106.0,46.400002,0.0,22.700001,87.099998,0.233911,8.886664,99.784554,0,0
169155,2025-11-27 16:38:01.694,106.0,46.400002,0.0,22.750000,86.700001,0.237196,8.886664,99.784554,0,0
169156,2025-11-27 16:38:46.754,106.0,46.400002,0.0,22.700001,89.000000,0.258948,8.886662,99.784546,0,0


In [76]:
# เรียงลำดับตาม devID และ timestamp
df = df.sort_values('timestamp').reset_index(drop=True)
df = df.set_index('timestamp')
#df['hour'] = df.index.hourผ
#df['day_of_week'] = df.index.dayofweek
df = df.reset_index() 

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168276 entries, 0 to 168275
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   timestamp  168276 non-null  datetime64[ns]
 1   devID      168276 non-null  float64       
 2   soil       168276 non-null  float64       
 3   rain       168276 non-null  float64       
 4   temp       168276 non-null  float64       
 5   humi       168276 non-null  float64       
 6   geo        168276 non-null  float64       
 7   lat        168276 non-null  float64       
 8   lng        168276 non-null  float64       
 9   iso_risk   168276 non-null  int64         
 10  lstm_risk  168276 non-null  int64         
dtypes: datetime64[ns](1), float64(8), int64(2)
memory usage: 14.1 MB


In [78]:
nan_counts = df.isna().sum()

nan_columns = nan_counts[nan_counts > 0]

print("📌 คอลัมน์ที่มีค่า NaN:")
print(nan_columns)

📌 คอลัมน์ที่มีค่า NaN:
Series([], dtype: int64)


In [79]:
#df = df.dropna(subset=['humi']).reset_index(drop=True)

In [80]:
nan_counts = df.isna().sum()
nan_columns = nan_counts[nan_counts > 0]
print("📌 คอลัมน์ที่มีค่า NaN:")
print(nan_columns)

📌 คอลัมน์ที่มีค่า NaN:
Series([], dtype: int64)


In [81]:
#df = df.dropna()
#
#nan_counts = df.isna().sum()
#nan_columns = nan_counts[nan_counts > 0]
#print("df102")
#print("📌 คอลัมน์ที่มีค่า NaN:")
#print(nan_columns)

In [82]:
#df[(df["iso_risk"]==2) & (df["lstm_risk"]==2)]

In [83]:
df.describe()

,timestamp,devID,soil,rain,temp,humi,geo,lat,lng,iso_risk,lstm_risk
count,168276,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000,168276.000000
mean,2025-11-10 17:37:29.208119552,107.126827,38.479539,0.412447,26.830935,85.649979,5.802132,9.316622,102.539911,0.079940,0.045128
min,2025-11-01 06:35:13.489000,105.000000,0.534899,0.000000,16.450001,33.750000,0.000000,-1.000000,-1.000000,0.000000,0.000000
25%,2025-11-06 10:39:33.303000064,106.000000,34.299999,0.000000,23.849999,77.599998,0.385413,8.866824,99.749580,0.000000,0.000000
50%,2025-11-10 12:13:51.344499968,107.000000,37.700001,0.000000,25.200001,89.750000,0.612796,8.868599,99.754539,0.000000,0.000000
75%,2025-11-14 04:00:50.000999936,109.000000,43.000000,0.000000,29.900000,95.799999,9.768927,8.886636,99.784538,0.000000,0.000000
max,2025-11-27 16:39:31.772000,109.000000,64.300003,45.681900,38.400002,100.000000,39.624933,8138.283203,23593.232420,2.000000,2.000000
std,NaN,1.442573,6.900702,2.418531,3.844727,12.438254,8.525796,58.061751,264.857615,0.346784,0.278708


In [84]:
def prepare_and_engineer_features(df):
    """
    ฟังก์ชันสำหรับเตรียมข้อมูล 1 แถวต่อ 1 นาที แยกตาม devID (Device ID)
    และสร้าง Feature Engineering โดยคำนวณค่า MA แยกรายเครื่อง
    """
    
    # 1. กำหนดกลุ่มคอลัมน์
    # คอลัมน์ข้อมูลดิบ (Sensor) -> ใช้ค่าเฉลี่ย (Mean)
    raw_cols = ['soil', 'rain', 'temp', 'humi', 'geo', 'lat', 'lng', 'iso_score', 'lstm_score', 'confidence']
    # คอลัมน์ฉลาก (Risk) -> ใช้ค่าสูงสุด (Max)
    risk_cols = ['iso_risk', 'lstm_risk']
    
    # ตรวจสอบว่าคอลัมน์มีอยู่จริงใน df หรือไม่
    raw_cols = [c for c in raw_cols if c in df.columns]
    risk_cols = [c for c in risk_cols if c in df.columns]

    # ฟังก์ชันย่อยสำหรับ process ข้อมูล **ทีละ devID**
    # ดังนั้นค่า Rolling/MA จะไม่ปนกันระหว่างเครื่องแน่นอน
    def process_group(g):
        if len(g) == 0:
            return None
            
        # ตั้ง index เป็น timestamp เพื่อใช้ resample
        if g.index.name != 'timestamp':
            g = g.set_index('timestamp')
        
        g = g.sort_index()

        # --- A. RESAMPLING (1 Minute) ---
        agg_dict = {}
        for c in raw_cols: agg_dict[c] = 'mean'
        for c in risk_cols: agg_dict[c] = 'max'
            
        # Resample เป็นราย 1 นาที (สร้าง row ใหม่ถ้าขาดหายไปในช่วงเวลา)
        g_resampled = g.resample('1T').agg(agg_dict)

        # --- B. MISSING VALUES ---
        # Interpolate เชื่อมค่าที่ขาดหาย (เฉพาะในเครื่องตัวเอง)
        g_resampled[raw_cols] = g_resampled[raw_cols].interpolate(method='time', limit_direction='both')
        g_resampled[raw_cols] = g_resampled[raw_cols].fillna(0)
        
        g_resampled[risk_cols] = g_resampled[risk_cols].fillna(0).astype(int)

        # --- C. FEATURE ENGINEERING (Calculate per Device) ---
        # การคำนวณ Rolling ตรงนี้ปลอดภัย เพราะ g_resampled มีแค่ข้อมูลของเครื่องเดียว
        
        # 1. Rain MA
        g_resampled['rain_ma_1h']  = g_resampled['rain'].rolling('1h', min_periods=1).mean()
        g_resampled['rain_ma_6h']  = g_resampled['rain'].rolling('6h', min_periods=1).mean()
        g_resampled['rain_ma_12h'] = g_resampled['rain'].rolling('12h', min_periods=1).mean()
        g_resampled['rain_ma_24h'] = g_resampled['rain'].rolling('24h', min_periods=1).mean()

        # 2. Soil MA
        g_resampled['soil_ma_4h']  = g_resampled['soil'].rolling('4h', min_periods=1).mean()
        g_resampled['soil_ma_8h']  = g_resampled['soil'].rolling('8h', min_periods=1).mean()
        g_resampled['soil_ma_12h'] = g_resampled['soil'].rolling('12h', min_periods=1).mean()
        g_resampled['soil_ma_16h'] = g_resampled['soil'].rolling('16h', min_periods=1).mean()

        # 3. Geo MA
        g_resampled['geo_ma_2m'] = g_resampled['geo'].rolling('2min', min_periods=1).mean()
        g_resampled['geo_ma_4m'] = g_resampled['geo'].rolling('4min', min_periods=1).mean()
        g_resampled['geo_ma_6m'] = g_resampled['geo'].rolling('6min', min_periods=1).mean()
        g_resampled['geo_ma_8m'] = g_resampled['geo'].rolling('8min', min_periods=1).mean()

        # --- D. ADDITIONAL FEATURES ---
        # Delta (ผลต่างกับนาทีก่อนหน้า)
        g_resampled['soil_diff_1m'] = g_resampled['soil'].diff().fillna(0)
        g_resampled['geo_diff_1m'] = g_resampled['geo'].diff().fillna(0)
        g_resampled['rain_diff_1m'] = g_resampled['rain'].diff().fillna(0)
        
        # Delta 1 hour
        g_resampled['soil_diff_1h'] = g_resampled['soil'].diff(periods=60).fillna(0)

        # Time Features
        g_resampled['hour_sin'] = np.sin(2 * np.pi * g_resampled.index.hour / 24)
        g_resampled['hour_cos'] = np.cos(2 * np.pi * g_resampled.index.hour / 24)

        return g_resampled

    # Group by devID แล้ว apply เพื่อให้ process แยกกันทีละเครื่อง
    # ผลลัพธ์จะได้ MultiIndex (devID, timestamp)
    df_processed = df.groupby('devID').apply(process_group)
    
    # Reset index เพื่อดึง devID กลับมาเป็น Column ปกติ
    df_processed = df_processed.reset_index(level=0) 

    return df_processed


df['timestamp'] = pd.to_datetime(df['timestamp'])
df_final = prepare_and_engineer_features(df)

C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\2463125305.py:35: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  g_resampled = g.resample('1T').agg(agg_dict)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\2463125305.py:35: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  g_resampled = g.resample('1T').agg(agg_dict)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\2463125305.py:35: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  g_resampled = g.resample('1T').agg(agg_dict)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\2463125305.py:35: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  g_resampled = g.resample('1T').agg(agg_dict)
C:\Users\ahmad\AppData\Local\Temp\ipykernel_3156\2463125305.py:35: FutureWarning: 'T' is deprecated and will be removed in a future version, ple

In [85]:
df_final = df_final.reset_index() 
df_final

,timestamp,devID,soil,rain,temp,humi,geo,lat,lng,iso_risk,...,geo_ma_2m,geo_ma_4m,geo_ma_6m,geo_ma_8m,soil_diff_1m,geo_diff_1m,rain_diff_1m,soil_diff_1h,hour_sin,hour_cos
0,2025-11-02 08:53:00,105.0,39.099998,0.0,29.500000,79.800003,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.866025,-0.5
1,2025-11-02 08:54:00,105.0,38.900002,0.0,29.600000,79.000000,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,0.000000,-0.199997,0.000000,0.0,0.000000,0.866025,-0.5
2,2025-11-02 08:55:00,105.0,38.900002,0.0,29.600000,79.299999,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.866025,-0.5
3,2025-11-02 08:56:00,105.0,38.900002,0.0,29.600000,79.599998,0.000000,0.000000,0.000000,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.866025,-0.5
4,2025-11-02 08:57:00,105.0,38.900002,0.0,29.500000,79.200001,0.166357,8.884827,99.783180,0,...,0.083178,0.041589,0.033271,0.033271,0.000000,0.166357,0.0,0.000000,0.866025,-0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148914,2025-11-20 16:35:00,109.0,43.799999,0.0,23.299999,98.800003,0.211447,8.874166,0.650000,0,...,0.248664,0.304638,0.323699,0.961002,0.025000,-0.074432,0.0,0.000000,-0.866025,-0.5
148915,2025-11-20 16:36:00,109.0,43.750000,0.0,23.299999,98.800003,0.342971,8.871380,50.199783,0,...,0.277209,0.300152,0.320507,0.535098,-0.049999,0.131523,0.0,-0.049999,-0.866025,-0.5
148916,2025-11-20 16:37:00,109.0,43.700001,0.0,23.299999,98.800003,0.474494,8.868594,99.749565,0,...,0.408732,0.328698,0.339336,0.344957,-0.049999,0.131523,0.0,-0.200001,-0.866025,-0.5
148917,2025-11-20 16:38:00,109.0,43.700001,0.0,23.299999,98.800003,0.452852,8.868595,99.749565,0,...,0.463673,0.370441,0.354659,0.356299,0.000000,-0.021642,0.0,-0.200001,-0.866025,-0.5


In [86]:
df_final['devID'].value_counts()

devID
108.0    37817
106.0    36593
107.0    27096
109.0    26636
105.0    20777
Name: count, dtype: int64

In [91]:

df_final_save = df_final[df_final['devID'].isin([109])]

save_path = f"dataMA/label_109_ma_prepared.csv"
df_final_save.to_csv(save_path, index=False)